# Notebook 6 — Envío de predicciones al PLC Beckhoff (ADS)

Este notebook toma el modelo entrenado en la Parte 2 y envía el resultado de la
clasificación al IPC Beckhoff mediante el protocolo ADS (librería `pyads`). Las lámparas
y la lógica del brazo se programan aparte, en TwinCAT.

El trabajo avanza en tres etapas:

1. Envío de variables booleanas, para comprobar la comunicación con el PLC.
2. Clasificación de una imagen y envío del color detectado.
3. Clasificación en vivo con la cámara (opcional).

Requisitos previos en TwinCAT: declarar las variables `bRojo`, `bAzul` y `bAmarillo` en
una GVL, construir un HMI con una lámpara por color y dejar el runtime en Run.

El código de comunicación con el PLC se entrega como referencia: no ha sido probado con el
hardware del laboratorio y puede requerir ajustes. Documentación de `pyads`:
https://pyads.readthedocs.io . Si no hay conexión, el notebook continúa en modo de
simulación e informa qué habría enviado.

## 1. Configuración

En la celda siguiente edite **el AMS Net ID de su IPC** y, si su GVL no se llama
`VARIABLES`, **el nombre de cada variable**. El resto del notebook no necesita cambios.

In [ ]:
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '-1')
import glob, time
import json as json_lib
from collections import deque
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

try:
    import pyads
    PYADS_OK = True
except ImportError:
    PYADS_OK = False

AMS_NET_ID = '5.80.201.232.1.1'    # reemplace por el AMS Net ID de su IPC
ADS_PORT = 851

VAR_POR_COLOR = {                  # nombre de cada BOOL en su GVL de TwinCAT
    'rojo': 'VARIABLES.bRojo',
    'azul': 'VARIABLES.bAzul',
    'amarillo': 'VARIABLES.bAmarillo',
}

UMBRAL = 0.60                      # confianza minima para enviar una prediccion

print('pyads disponible' if PYADS_OK else 'pyads no instalado: modo simulacion')

## 2. Conexión con el PLC

Requiere TwinCAT en Run, el AMS Net ID correcto y una ruta ADS entre la PC y el IPC. Si la
ruta no existe, ADS devuelve el error `Missing ADS routes (7)`; se crea en TwinCAT, en
Router → Edit Routes → Add Route.

In [ ]:
plc = {'conn': None, 'connected': False}

def conectar():
    if not PYADS_OK:
        print('pyads no instalado: modo simulacion')
        return
    try:
        conn = pyads.Connection(AMS_NET_ID, ADS_PORT)
        conn.open()
        conn.read_state()                  # confirma que la ruta ADS responde
        plc['conn'] = conn
        plc['connected'] = True
        print('Conectado a', AMS_NET_ID)
    except Exception as e:
        plc['connected'] = False
        print('Sin conexion (', e, '): modo simulacion')

conectar()

## 3. Envío de variables booleanas

Primera comprobación: enviar un valor al PLC y verlo en el HMI. `enviar_color` pone en
`True` la variable del color indicado y en `False` las otras dos; `apagar_todo` las deja
todas en `False`. Ejecute las líneas finales y observe la lámpara correspondiente en
TwinCAT.

In [ ]:
def escribir_bool(variable, valor):
    if not plc['connected']:
        return
    try:
        plc['conn'].write_by_name(variable, bool(valor), pyads.PLCTYPE_BOOL)
    except Exception as e:
        print('Error ADS:', e)
        plc['connected'] = False

def enviar_color(color):
    for c, variable in VAR_POR_COLOR.items():
        escribir_bool(variable, c == color)
    destino = 'PLC' if plc['connected'] else 'simulacion'
    print('Enviado a', destino + ':', color if color else 'todo en False')

def apagar_todo():
    enviar_color(None)


enviar_color('rojo')
# enviar_color('azul')
# enviar_color('amarillo')
# apagar_todo()

## 4. Clasificación de una imagen y envío

Se carga el último modelo entrenado y se clasifica una imagen. El color se envía solo si
su confianza supera `UMBRAL`; la clase `fondo` no tiene variable asociada y nunca se envía.

In [ ]:
rutas = glob.glob('modelos/*.h5') or glob.glob('../modelos/*.h5')
MODEL_PATH = max(rutas, key=os.path.getmtime)
modelo = load_model(MODEL_PATH, compile=False)
info = json_lib.load(open(MODEL_PATH.rsplit('.', 1)[0] + '.json'))
CLASES = info['class_names']
IMG_SIZE = info['img_size']
PREP = info['preprocessing']
print('Modelo:', os.path.basename(MODEL_PATH), '-', CLASES)

def preparar(imagen_rgb):
    x = imagen_rgb.astype('float32')
    return preprocess_input(x) if PREP == 'mobilenet' else x / 255.0

def predecir(ruta, mostrar=True):
    img = load_img(ruta, target_size=(IMG_SIZE, IMG_SIZE))
    pred = modelo.predict(np.expand_dims(preparar(img_to_array(img)), 0), verbose=0)[0]
    i = int(np.argmax(pred))
    if mostrar:
        plt.imshow(img); plt.axis('off'); plt.title(CLASES[i] + f' ({pred[i]:.0%})'); plt.show()
    return CLASES[i], float(pred[i])

def enviar_prediccion(ruta):
    clase, conf = predecir(ruta)
    if clase not in VAR_POR_COLOR:
        print('No se envia:', clase, 'no tiene variable asociada')
    elif conf < UMBRAL:
        print('No se envia: confianza', f'{conf:.0%}', 'menor que el umbral', f'{UMBRAL:.0%}')
    else:
        enviar_color(clase)

In [ ]:
# Imagen de prueba (cambie la ruta por una foto propia si lo desea)
ejemplos = glob.glob('data/tapitas/rojo/*') or glob.glob('../data/tapitas/rojo/*')
enviar_prediccion(ejemplos[0])

## 5. Clasificación en vivo con la cámara (opcional)

La cámara clasifica de forma continua y muestra la predicción. El envío al PLC ocurre solo
al pulsar la barra espaciadora; con `q` se cierra. En WSL2 no hay acceso a la cámara: este
paso se ejecuta desde Windows.

In [ ]:
def es_wsl():
    try:
        return 'microsoft' in open('/proc/version').read().lower()
    except Exception:
        return False

if es_wsl():
    print('Sin camara en WSL2; ejecute este paso desde Windows.')
else:
    cam = cv2.VideoCapture(0)              # use 1 si no detecta la camara
    historial = deque(maxlen=5)            # promedio de los ultimos frames
    ultimo = 'ninguno'
    print('Camara abierta. Barra espaciadora: enviar. q: salir.')
    try:
        while True:
            ok, frame = cam.read()
            if not ok:
                break
            rgb = cv2.cvtColor(cv2.resize(frame, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_BGR2RGB)
            pred = modelo.predict(np.expand_dims(preparar(rgb), 0), verbose=0)[0]
            historial.append(pred)
            media = np.mean(historial, axis=0)
            clase = CLASES[int(np.argmax(media))]
            conf = float(np.max(media))

            cv2.putText(frame, clase + f' ({conf*100:.0f}%)', (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.putText(frame, 'Ultimo envio: ' + ultimo, (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 2)
            cv2.imshow('Camara', frame)

            tecla = cv2.waitKey(1) & 0xFF
            if tecla == ord('q'):
                break
            if tecla == ord(' '):
                if clase in VAR_POR_COLOR and conf >= UMBRAL:
                    enviar_color(clase)
                    ultimo = clase
                else:
                    enviar_color(None)
                    ultimo = 'ninguno'
    finally:
        apagar_todo()
        cam.release()
        cv2.destroyAllWindows()
        print('Camara cerrada.')